# Code mẫu — Giảm chiều

PCA học từ dữ liệu một phép chiếu xuống ít chiều hơn. Phép chiếu ấy giữ được nhiều phương sai nhất và tái tạo lại dữ liệu ít sai nhất, nhưng nó không hề biết nhãn. Ba ví dụ dưới đây theo dõi shape của các ma trận, xem phần nào bị mất khi nén, và đo ảnh hưởng của việc nén lên một bài dự đoán.

## Ví dụ 1: Nén hai phép đo thành một tọa độ

Bốn mẫu có hai phép đo cùng đơn vị: `[1, 1]`, `[2, 2]`, `[3, 2]`, `[4, 3]`. Hai phép đo này tăng gần như cùng nhau. Ta trừ mean của từng cột, tính PCA bằng SVD, rồi tái tạo lại dữ liệu từ một tọa độ duy nhất, để thấy tận mắt phần bị bỏ đi.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

X = np.array([[1., 1.], [2., 2.], [3., 2.], [4., 3.]])
mean = X.mean(axis=0)
centered = X - mean
_, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
direction = Vt[0].copy()
if direction[0] < 0:  # Lật dấu cho dễ đối chiếu. Hướng ngược dấu vẫn là cùng một trục.
    direction *= -1
coordinates = centered @ direction[:, None]
reconstructed = coordinates @ direction[None, :] + mean
variance_ratio = singular_values[0]**2 / np.sum(singular_values**2)
library = PCA(n_components=1, svd_solver="full").fit(X)
print("mean / hướng:", mean.round(3).tolist(), direction.round(3).tolist())
print("shape tọa độ / tái tạo:", coordinates.shape, reconstructed.shape)
print("tọa độ:", coordinates.ravel().round(3).tolist())
print("tái tạo:", reconstructed.round(3).tolist())
print("variance giữ / MSE tái tạo:", round(float(variance_ratio), 4),
      round(float(np.mean((X - reconstructed)**2)), 4))
print("khớp thư viện:", np.allclose(
    reconstructed, library.inverse_transform(library.transform(X))
))
assert np.allclose(reconstructed, library.inverse_transform(library.transform(X)))

mean / hướng: [2.5, 2.0] [0.851, 0.526]
shape tọa độ / tái tạo: (4, 1) (4, 2)
tọa độ: [-1.802, -0.425, 0.425, 1.802]
tái tạo: [[0.967, 1.053], [2.138, 1.776], [2.862, 2.224], [4.033, 2.947]]
variance giữ / MSE tái tạo: 0.9792 0.0182
khớp thư viện: True


Mean là `[2.5, 2]`, và PCA tìm hướng mà dữ liệu trải rộng nhất quanh điểm tâm ấy. Tọa độ có shape `(4, 1)`, tức vẫn đủ bốn mẫu nhưng mỗi mẫu chỉ còn một con số.

Một chiều này giữ được 97.92% phương sai. Dù vậy, mẫu `[2, 2]` khi tái tạo chỉ còn khoảng `[2.138, 1.776]`. Phần lệch ấy chính là phần nằm theo hướng thứ hai, hướng đã bị bỏ. Thư viện có thể trả về hướng ngược dấu, và khi đó mọi tọa độ cũng đổi dấu. Vì vậy ta so dữ liệu tái tạo với thư viện, chứ không so tọa độ.

## Ví dụ 2: Giữ gần hết phương sai mà vẫn mất sạch thông tin về nhãn

Cột thứ nhất dao động rất mạnh nhưng không liên quan gì tới lớp. Cột thứ hai chỉ là −0.1 hoặc +0.1, nhưng chính dấu của nó quyết định lớp. Dữ liệu được dựng đối xứng để dễ kiểm bằng tay: cột lớn có phương sai 80, cột nhỏ chỉ có 0.01. Nén xuống một chiều, PCA sẽ chọn cột lớn.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X_train = np.array([[large, small] for large in [-12., -4., 4., 12.]
                   for small in [-0.1, 0.1]])
y_train = (X_train[:, 1] > 0).astype(int)
X_valid = np.array([[large, small] for large in [-10., -2., 2., 10.]
                   for small in [-0.1, 0.1]])
y_valid = (X_valid[:, 1] > 0).astype(int)
for dimensions in [1, 2]:
    model = make_pipeline(PCA(n_components=dimensions, svd_solver="full"),
                          LogisticRegression(C=1000., max_iter=500))
    model.fit(X_train, y_train)
    pca = model[0]
    print("k =", dimensions,
          "variance giữ:", round(float(pca.explained_variance_ratio_.sum()), 6),
          "validation accuracy:", round(float(model.score(X_valid, y_valid)), 3))
    print("nhãn đoán:", model.predict(X_valid).tolist())
assert np.isclose(X_train[:, 0].var(), 80.)
assert np.isclose(X_train[:, 1].var(), 0.01)

k = 1 variance giữ: 0.999875 validation accuracy: 0.5
nhãn đoán: [0, 0, 0, 0, 0, 0, 0, 0]
k = 2 variance giữ: 1.0 validation accuracy: 1.0
nhãn đoán: [0, 1, 0, 1, 0, 1, 0, 1]


Một chiều giữ 99.9875% phương sai, nhưng accuracy chỉ còn 0.5, ngang với đoán bừa. Hai mẫu có cùng giá trị ở cột lớn mà nhãn trái nhau sẽ bị nén thành cùng một tọa độ. Từ đó trở đi, không bộ phân loại nào tách được chúng nữa. Giữ đủ hai chiều thì dấu của cột nhỏ quay lại, và accuracy lên 1.

Bài học ở đây: giữ 99.99% phương sai không có nghĩa là giữ được 99.99% khả năng dự đoán. Số chiều phải được chọn theo việc ta định làm sau khi chiếu.

## Ví dụ 3: Chọn số chiều cho ảnh chữ số bằng Pipeline

Digits là bộ ảnh chữ số viết tay cỡ 8×8 có sẵn trong scikit-learn, tức 64 pixel cho mỗi ảnh. Ta lấy cố định 700 ảnh để chạy nhanh trên CPU, tách riêng test, rồi dùng ba fold trên phần còn lại để chọn giữa 8, 16 và 32 chiều. Mỗi pixel nhận giá trị từ 0 đến 16, nên ta chia cho 16. Con số 16 lấy từ mô tả của bộ dữ liệu, không học từ test.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

data = load_digits()
selected = np.random.default_rng(29).permutation(len(data.target))[:700]
X, y = data.data[selected] / 16., data.target[selected]
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2
)
pipeline = Pipeline([
    ("pca", PCA(svd_solver="full")),
    ("classifier", LogisticRegression(max_iter=500)),
])
folds = StratifiedKFold(3, shuffle=True, random_state=3)
search = GridSearchCV(pipeline, {"pca__n_components": [8, 16, 32]},
                      cv=folds, scoring="accuracy", n_jobs=1)
search.fit(X_dev, y_dev)
print("CV accuracy cho 8,16,32 chiều:",
      search.cv_results_["mean_test_score"].round(3).tolist())
print("số chiều chọn:", search.best_params_["pca__n_components"])

CV accuracy cho 8,16,32 chiều: [0.895, 0.945, 0.941]
số chiều chọn: 16


Vì PCA nằm trong Pipeline, nó được fit lại trên phần train của từng fold. Nếu fit PCA một lần trên cả tập dev rồi mới chạy CV, phần validation của mỗi fold đã góp vào phép chiếu. Ở đây CV chọn số chiều theo accuracy phân loại, khác với cách chọn theo một ngưỡng phương sai như 95%. Tăng từ 8 lên 16 chiều giúp rõ rệt, còn từ 16 lên 32 thì không giúp thêm.

Giờ lấy model đã chọn, nén tập test, tái tạo lại, và chấm điểm.

In [ ]:
chosen = search.best_estimator_
pca = chosen["pca"]
compressed = pca.transform(X_test)
restored = pca.inverse_transform(compressed)
pred = chosen.predict(X_test)
print("shape trước / sau:", X_test.shape, compressed.shape)
print("variance giữ trên train:", round(float(pca.explained_variance_ratio_.sum()), 3))
print("test accuracy:", round(accuracy_score(y_test, pred), 3))
print("4 nhãn thật / đoán:", y_test[:4].tolist(), pred[:4].tolist())
print("MSE tái tạo test:", round(float(np.mean((X_test - restored)**2)), 5))
assert pca.n_samples_ == len(X_dev)
assert compressed.shape[1] == search.best_params_["pca__n_components"]

shape trước / sau: (175, 64) (175, 16)
variance giữ trên train: 0.855
test accuracy: 0.954
4 nhãn thật / đoán: [8, 0, 7, 1] [8, 0, 7, 1]
MSE tái tạo test: 0.01198


Mỗi ảnh test đi từ 64 số xuống còn 16 số. Mười sáu chiều này chỉ giữ 85.5% phương sai, nhưng test accuracy vẫn đạt 0.954. Hai con số không cần bằng nhau, vì chúng trả lời hai câu hỏi khác nhau. MSE tái tạo 0.01198 đo phần ảnh bị mất, tính trên thang pixel đã chia 16.

Đã xem test rồi thì không quay lại đổi số chiều nữa. Nếu muốn thử chuẩn hóa từng pixel trước PCA, hãy nhớ rằng làm vậy là đổi hẳn bài toán PCA, nên phải so riêng trên validation.